In [25]:
import geopandas as gpd
from pathlib import Path

# Path to the GeoPackage
path = "data/raw/Geopackage_2021_G02_AUST_GDA2020/G02_AUST_GDA2020.gpkg"

# Load SA2 layer
gdf = gpd.read_file(path, layer="G02_SA2_2021_AUST")

# Keep only VIC (SA2 codes starting with '2')
gdf_vic = gdf[gdf["SA2_CODE_2021"].astype(str).str.startswith("2")]

# Select useful columns
income = gdf_vic[[
    "SA2_CODE_2021",
    "SA2_NAME_2021",
    "Median_tot_hhd_inc_weekly"
]].rename(columns={
    "SA2_CODE_2021": "sa2_code",
    "SA2_NAME_2021": "sa2_name",
    "Median_tot_hhd_inc_weekly": "median_hh_income"
})

print(income.head())

# Save cleaned CSV
Path("data/external").mkdir(parents=True, exist_ok=True)
income.to_csv("data/external/income_sa2.csv", index=False)

      sa2_code       sa2_name  median_hh_income
644  201011001      Alfredton            1952.0
645  201011002       Ballarat            1573.0
646  201011005      Buninyong            1927.0
647  201011006      Delacombe            1627.0
648  201011007  Smythes Creek            2065.0


In [45]:
income[income["sa2_name"].str.contains("Broadmeadows", case=False)]

,sa2_code,sa2_name,median_hh_income
927,210051242,Broadmeadows,1172.0


In [47]:
import geopandas as gpd
import pandas as pd
import re
from pathlib import Path

# === 1) Paths ===
gpkg = "data/raw/Geopackage_2021_G33_AUST_GDA2020/G33_AUST_GDA2020.gpkg"
layer = "G33_SA2_2021_AUST"  # SA2-level G33 layer

# === 2) Load SA2 table ===
gdf = gpd.read_file(gpkg, layer=layer)

# Keep only VIC SA2s (codes starting with '2')
gdf_vic = gdf[gdf["SA2_CODE_2021"].astype(str).str.startswith("2")].copy()

# We'll keep identifiers up front
id_cols = ["SA2_CODE_2021", "SA2_NAME_2021"]
non_id_cols = [c for c in gdf_vic.columns if c not in id_cols and c != gdf_vic.geometry.name]

# === 3) Identify income bracket columns (any composition, totals) ===
# Heuristics:
# - bracket range cols look like "... 1_149 ...", "150_299", ..., "2000_2999"
# - open-ended high bracket like "3000_plus|3000_more|3000_and_over"
# - negative/nil like "Neg" or "Nil"
# - we prefer composition totals, which typically end with "_Tot" (but we'll fall back if needed)

def is_total_col(col):
    return col.endswith("_Tot") or col.endswith("_tot") or col.lower().endswith(" total") or col.lower().endswith("_total")

range_pat = re.compile(r"(?P<lo>\d{1,4})_(?P<hi>\d{1,4})")
high_open_pat = re.compile(r"(3000)(?!_\d)")  # 3000 with no upper bound
neg_nil_pat = re.compile(r"(neg|nil)", re.IGNORECASE)

# pick numeric distribution columns
cand_cols = []
for c in non_id_cols:
    if gdf_vic[c].dtype.kind in "if":  # numeric
        text = c.replace("__", "_")
        if range_pat.search(text) or high_open_pat.search(text) or neg_nil_pat.search(text):
            cand_cols.append(c)

# prefer totals across compositions if available
tot_cols = [c for c in cand_cols if is_total_col(c)]
dist_cols = tot_cols if len(tot_cols) >= 10 else cand_cols  # fallback

# Persist column list for transparency (optional)
Path("data/external").mkdir(parents=True, exist_ok=True)
pd.Series(dist_cols, name="g33_dist_columns").to_csv("data/external/G33_columns_used.csv", index=False)

# === 4) Split columns into low (<$650) vs high (≥$2000) ===
low_cols, high_cols = [], []
for c in dist_cols:
    s = c
    if neg_nil_pat.search(s):
        low_cols.append(c)
        continue
    m = range_pat.search(s)
    if m:
        lo, hi = int(m.group("lo")), int(m.group("hi"))
        if hi <= 649:
            low_cols.append(c)
        if lo >= 2000:
            high_cols.append(c)
        continue
    if high_open_pat.search(s):  # "3000+" style
        high_cols.append(c)

# If we somehow picked no columns, raise a helpful error
if not low_cols or not high_cols:
    raise RuntimeError(
        "Could not auto-detect low/high income columns in G33. "
        "Check data/external/G33_columns_used.csv and adjust patterns."
    )

# === 5) Row-wise sums & percentages ===
g33 = gdf_vic[id_cols + dist_cols].copy()

# total households across all brackets (summing all bracket totals)
g33["hh_total"] = g33[dist_cols].sum(axis=1, skipna=True)

g33["hh_low_u650"] = g33[low_cols].sum(axis=1, skipna=True)
g33["hh_high_ge2000"] = g33[high_cols].sum(axis=1, skipna=True)

# Avoid divide-by-zero
g33["pct_low_u650"] = (g33["hh_low_u650"] / g33["hh_total"]).where(g33["hh_total"] > 0)
g33["pct_high_ge2000"] = (g33["hh_high_ge2000"] / g33["hh_total"]).where(g33["hh_total"] > 0)

# === 6) Final tidy output ===
out = g33.rename(columns={
    "SA2_CODE_2021": "sa2_code",
    "SA2_NAME_2021": "sa2_name"
})[["sa2_code", "sa2_name", "hh_total", "hh_low_u650", "hh_high_ge2000", "pct_low_u650", "pct_high_ge2000"]]

out.to_csv("data/external/income_dist_sa2.csv", index=False)
out.head()

,sa2_code,sa2_name,hh_total,hh_low_u650,hh_high_ge2000,pct_low_u650,pct_high_ge2000
644,201011001,Alfredton,4770,651,2013,0.136478,0.422013
645,201011002,Ballarat,4091,904,1220,0.220973,0.298216
646,201011005,Buninyong,2046,320,828,0.156403,0.404692
647,201011006,Delacombe,3457,525,1193,0.151866,0.345097
648,201011007,Smythes Creek,1130,148,509,0.130973,0.450442


In [49]:
import pandas as pd

# Load both processed datasets
g02 = pd.read_csv("data/external/income_sa2.csv")       # from G02
g33 = pd.read_csv("data/external/income_dist_sa2.csv")  # from G33

# Merge on SA2 code
income_combined = g02.merge(g33, on=["sa2_code", "sa2_name"], how="inner")

# Inspect
print(income_combined.head())
print("Shape:", income_combined.shape)

# Save to external
income_combined.to_csv("data/external/income_features_sa2.csv", index=False)

    sa2_code       sa2_name  median_hh_income  hh_total  hh_low_u650  \
0  201011001      Alfredton            1952.0      4770          651   
1  201011002       Ballarat            1573.0      4091          904   
2  201011005      Buninyong            1927.0      2046          320   
3  201011006      Delacombe            1627.0      3457          525   
4  201011007  Smythes Creek            2065.0      1130          148   

   hh_high_ge2000  pct_low_u650  pct_high_ge2000  
0            2013      0.136478         0.422013  
1            1220      0.220973         0.298216  
2             828      0.156403         0.404692  
3            1193      0.151866         0.345097  
4             509      0.130973         0.450442  
Shape: (524, 8)


In [51]:
# READ ME

Source

This dataset combines two tables from the Australian Bureau of Statistics (ABS) 2021 Census – General Community Profile (GCP):

G02 – Selected medians and averages

G33 – Total household income (weekly) by household composition

Extracted at SA2 geography (Statistical Area Level 2) using ABS GeoPackages:

G02_AUST_GDA2020.gpkg

G33_AUST_GDA2020.gpkg

Processed into a single CSV for Victoria (state code 2).
                                          
Description

The dataset provides both central tendency (median) and distributional (low/high-income shares) measures of household income at the SA2 level.

These indicators give a richer picture of local affordability and socio-economic conditions.

Fields in income_features_sa2.csv
Column	Description	Example
sa2_code - ABS Statistical Area Level 2 code (2021 ASGS)	201011001
sa2_name - SA2 area name	Alfredton
median_hh_income - Median weekly household income (from G02)	1952
hh_total - Total number of households (from G33)	4770
hh_low_u650 - Number of households earning less than $650 per week (from G33)	651
hh_high_ge2000 - Number of households earning $2000 or more per week (from G33)	2013
pct_low_u650 - Proportion of low-income households = hh_low_u650 / hh_total	0.136
pct_high_ge2000 - Proportion of high-income households = hh_high_ge2000 / hh_total	0.422

In [65]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# ---- Paths ----
gpkg_2016 = "data/raw/Geopackage_2016_HIHC_for_AUST/census2016_hihc_aus_short.gpkg"
layer_2016 = "census2016_hihc_aus_sa2_short"   # from listlayers()

# ---- Read 2016 SA2 layer ----
gdf16 = gpd.read_file(gpkg_2016, layer=layer_2016)

# Normalise column names: lowercase + strip spaces
gdf16.columns = [c.strip().lower() for c in gdf16.columns]

# Helper to pick a column by preferred candidates or fallback pattern
def pick(colnames, *cands, startswith=None, contains_all=None):
    s = set(colnames)
    for c in cands:
        if c and c in s:
            return c
    if startswith:
        for c in colnames:
            if c.startswith(startswith):
                return c
    if contains_all:
        for c in colnames:
            if all(k in c for k in contains_all):
                return c
    raise KeyError(f"Could not find a column among candidates: {cands} or pattern: {startswith or contains_all}")

cols = list(gdf16.columns)

# 1) SA2 code column (varies across packs)
sa2_code_col = pick(
    cols,
    "sa2_maincode_2016", "sa2_code_2016",
    startswith="sa2_"
)

# 2) Median weekly household income column
#    In your printout it's 'median_tot_hhd_inc_weekly'
median_col = None
candidates = [
    "median_tot_hhd_inc_weekly",
    "median_total_household_income_weekly",
]
for c in candidates:
    if c in cols:
        median_col = c
        break
if median_col is None:
    # generic fallback: must contain median + (hhd|household) + inc + (week|weekly)
    for c in cols:
        if ("median" in c) and (("hhd" in c) or ("household" in c)) and ("inc" in c) and (("week" in c) or ("weekly" in c)):
            median_col = c
            break
if median_col is None:
    raise KeyError(f"Couldn't detect median household income column. Columns include e.g.: {cols[:20]} ...")

# ---- Filter VIC (SA2 codes start with '2') ----
gdf16_vic = gdf16[gdf16[sa2_code_col].astype(str).str.startswith("2")].copy()

# ---- Build tidy output ----
income_2016 = gdf16_vic[[sa2_code_col, median_col]].rename(
    columns={
        sa2_code_col: "sa2_code",
        median_col: "median_hh_income_2016"
    }
)

# ---- Save tidy CSV ----
Path("data/external").mkdir(parents=True, exist_ok=True)
income_2016.to_csv("data/external/income_sa2_2016.csv", index=False)

# ---- Sanity checks ----
print(income_2016.head())
print(income_2016["median_hh_income_2016"].describe())
print("Rows (VIC SA2s):", len(income_2016))

     sa2_code  median_hh_income_2016
4   210011227                      0
10  210031438                   1254
12  209041216                   1432
15  213031349                   1765
26  212021297                   1297
count     464.000000
mean     1431.338362
std       420.316855
min         0.000000
25%      1127.500000
50%      1411.500000
75%      1723.000000
max      2438.000000
Name: median_hh_income_2016, dtype: float64
Rows (VIC SA2s): 464


In [67]:
import pandas as pd
from pathlib import Path

# --- Load existing combined (G02 2021 + G33 2021) ---
features_2021 = pd.read_csv("data/external/income_features_sa2.csv")
# Expected cols include: sa2_code, sa2_name, median_hh_income, hh_total, pct_low_u650, pct_high_ge2000, ...

# --- Load 2016 medians ---
inc2016 = pd.read_csv("data/external/income_sa2_2016.csv")

# Clean 2016 medians: treat zeros / negatives as missing
inc2016["median_hh_income_2016"] = pd.to_numeric(inc2016["median_hh_income_2016"], errors="coerce")
inc2016.loc[inc2016["median_hh_income_2016"] <= 0, "median_hh_income_2016"] = pd.NA

# --- Merge on SA2 code (keep 2021 names) ---
merged = (
    features_2021
    .merge(inc2016[["sa2_code", "median_hh_income_2016"]], on="sa2_code", how="left")
)

# --- Compute growth (2016 → 2021) ---
merged["income_growth_rate"] = (
    (merged["median_hh_income"] - merged["median_hh_income_2016"])
    / merged["median_hh_income_2016"]
)

# Optional: quick sanity checks
print(merged[["sa2_code","sa2_name","median_hh_income_2016","median_hh_income","income_growth_rate"]].head())
print("Rows:", len(merged), "| Missing 2016 medians:", merged["median_hh_income_2016"].isna().sum())

# --- Save (overwrite) ---
Path("data/external").mkdir(parents=True, exist_ok=True)
# (Optional) backup the pre-growth version once
merged.to_csv("data/external/income_features_sa2.csv", index=False)
print("Saved -> data/external/income_features_sa2.csv")

    sa2_code       sa2_name  median_hh_income_2016  median_hh_income  \
0  201011001      Alfredton                 1585.0            1952.0   
1  201011002       Ballarat                 1327.0            1573.0   
2  201011005      Buninyong                 1634.0            1927.0   
3  201011006      Delacombe                 1272.0            1627.0   
4  201011007  Smythes Creek                 1583.0            2065.0   

   income_growth_rate  
0            0.231546  
1            0.185381  
2            0.179315  
3            0.279088  
4            0.304485  
Rows: 524 | Missing 2016 medians: 121
Saved -> data/external/income_features_sa2.csv


In [71]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# 1) Read G02 and pull Median age at SA2 (VIC only)
g02_path = "data/raw/Geopackage_2021_G02_AUST_GDA2020/G02_AUST_GDA2020.gpkg"
g02 = gpd.read_file(g02_path, layer="G02_SA2_2021_AUST")

demo = (
    g02[g02["SA2_CODE_2021"].astype(str).str.startswith("2")]  # Victoria
      [["SA2_CODE_2021", "SA2_NAME_2021", "Median_age_persons"]]
      .rename(columns={
          "SA2_CODE_2021": "sa2_code",
          "SA2_NAME_2021": "sa2_name",
          "Median_age_persons": "median_age"
      })
)

# 2) Load existing income features
features = pd.read_csv("data/external/income_features_sa2.csv")

# ---- Make merge keys the same dtype (fix) ----
features["sa2_code"] = features["sa2_code"].astype(str)
demo["sa2_code"] = demo["sa2_code"].astype(str)

# 3) Merge and save
merged = features.merge(demo[["sa2_code", "median_age"]], on="sa2_code", how="left")

Path("data/external").mkdir(parents=True, exist_ok=True)
merged.to_csv("data/external/income_demo_sa2.csv", index=False)

merged.head()

,sa2_code,sa2_name,median_hh_income,hh_total,hh_low_u650,hh_high_ge2000,pct_low_u650,pct_high_ge2000,median_hh_income_2016,income_growth_rate,median_age
0,201011001,Alfredton,1952.0,4770,651,2013,0.136478,0.422013,1585.0,0.231546,34.0
1,201011002,Ballarat,1573.0,4091,904,1220,0.220973,0.298216,1327.0,0.185381,45.0
2,201011005,Buninyong,1927.0,2046,320,828,0.156403,0.404692,1634.0,0.179315,42.0
3,201011006,Delacombe,1627.0,3457,525,1193,0.151866,0.345097,1272.0,0.279088,33.0
4,201011007,Smythes Creek,2065.0,1130,148,509,0.130973,0.450442,1583.0,0.304485,41.0


In [85]:
import geopandas as gpd
import pandas as pd
import re
from pathlib import Path

# --- Paths ---
g04a_path = "data/raw/Geopackage_2021_G04_VIC_GDA2020/G04A_VIC_GDA2020.gpkg"
g04b_path = "data/raw/Geopackage_2021_G04_VIC_GDA2020/G04B_VIC_GDA2020.gpkg"
layer_a = "G04A_SA2_2021_VIC"
layer_b = "G04B_SA2_2021_VIC"

# --- Read both packs ---
ga = gpd.read_file(g04a_path, layer=layer_a)
gb = gpd.read_file(g04b_path, layer=layer_b)

# Normalize cols
ga.columns = [c.strip().lower() for c in ga.columns]
gb.columns = [c.strip().lower() for c in gb.columns]

# SA2 ids
code_col_a = next(c for c in ga.columns if c.startswith("sa2_code"))
name_col_a = next(c for c in ga.columns if c.startswith("sa2_name"))
code_col_b = next(c for c in gb.columns if c.startswith("sa2_code"))
name_col_b = next(c for c in gb.columns if c.startswith("sa2_name"))

# Keep VIC (already VIC pack but guard anyway)
ga = ga[ga[code_col_a].astype(str).str.startswith("2")].copy()
gb = gb[gb[code_col_b].astype(str).str.startswith("2")].copy()

# --- Merge on SA2 code (outer, to capture all bands) ---
ga["sa2_code"] = ga[code_col_a].astype(str)
gb["sa2_code"] = gb[code_col_b].astype(str)

# drop duplicate name cols from gb before merge
to_drop_b = [name_col_b] + [c for c in gb.columns if c in ("geometry", "geom") or c == code_col_b]
gb_small = gb.drop(columns=[c for c in to_drop_b if c in gb.columns], errors="ignore")

g = ga.merge(gb_small, on="sa2_code", how="outer", validate="one_to_one")
g.rename(columns={name_col_a: "sa2_name"}, inplace=True)

# --- Collect all 5-year PERSONS columns across A+B ---
cols = list(g.columns)
five_year_p_cols = [c for c in cols if re.fullmatch(r"age_yr_\d+_\d+_p", c)]
five_year_p_cols += [c for c in cols if re.fullmatch(r"age_yr_85.*_p", c)]
five_year_p_cols = sorted(set(five_year_p_cols))

# Helper to pick available columns for a set of desired bands
def take_available(desired):
    return [c for c in desired if c in five_year_p_cols]

group_0_14  = take_available(["age_yr_0_4_p","age_yr_5_9_p","age_yr_10_14_p"])
group_15_34 = take_available(["age_yr_15_19_p","age_yr_20_24_p","age_yr_25_29_p","age_yr_30_34_p"])
group_35_64 = take_available(["age_yr_35_39_p","age_yr_40_44_p","age_yr_45_49_p","age_yr_50_54_p","age_yr_55_59_p","age_yr_60_64_p"])
group_65p   = [c for c in five_year_p_cols if any(k in c for k in ["_65_69_", "_70_74_", "_75_79_", "_80_84_", "_85_"])]

# Optional: print what we found
print("Bands present:")
print(" 0–14 :", group_0_14)
print(" 15–34:", group_15_34)
print(" 35–64:", group_35_64)
print(" 65+  :", group_65p)

# --- Totals & shares ---
g["persons_total"] = g[five_year_p_cols].sum(axis=1, skipna=True)

g["cnt_0_14"]    = g[group_0_14].sum(axis=1, skipna=True) if group_0_14 else 0
g["cnt_15_34"]   = g[group_15_34].sum(axis=1, skipna=True) if group_15_34 else 0
g["cnt_35_64"]   = g[group_35_64].sum(axis=1, skipna=True) if group_35_64 else 0
g["cnt_65_plus"] = g[group_65p].sum(axis=1, skipna=True)   if group_65p   else 0

den = g["persons_total"].replace({0: pd.NA})
g["pct_0_14"]    = g["cnt_0_14"]    / den
g["pct_15_34"]   = g["cnt_15_34"]   / den
g["pct_35_64"]   = g["cnt_35_64"]   / den
g["pct_65_plus"] = g["cnt_65_plus"] / den

# --- Tidy & save ---
out = g[["sa2_code","sa2_name",
          "cnt_0_14","cnt_15_34","cnt_35_64","cnt_65_plus",
          "pct_0_14","pct_15_34","pct_35_64","pct_65_plus"]].copy()

Path("data/external").mkdir(parents=True, exist_ok=True)
out.to_csv("data/external/age_dist_sa2.csv", index=False)
out.head()

Bands present:
 0–14 : ['age_yr_0_4_p', 'age_yr_5_9_p', 'age_yr_10_14_p']
 15–34: ['age_yr_15_19_p', 'age_yr_20_24_p', 'age_yr_25_29_p', 'age_yr_30_34_p']
 35–64: ['age_yr_35_39_p', 'age_yr_40_44_p', 'age_yr_45_49_p', 'age_yr_50_54_p', 'age_yr_55_59_p', 'age_yr_60_64_p']
 65+  : ['age_yr_65_69_p', 'age_yr_70_74_p', 'age_yr_75_79_p', 'age_yr_80_84_p', 'age_yr_85_89_p']


,sa2_code,sa2_name,cnt_0_14,cnt_15_34,cnt_35_64,cnt_65_plus,pct_0_14,pct_15_34,pct_35_64,pct_65_plus
0,201011001,Alfredton,4032,4462,6167,2120,0.239473,0.265012,0.366277,0.125913
1,201011002,Ballarat,1783,2870,4648,2668,0.147064,0.236721,0.383372,0.220059
2,201011005,Buninyong,1508,1514,2907,1304,0.207713,0.20854,0.400413,0.179614
3,201011006,Delacombe,2273,3376,3431,1487,0.213327,0.316847,0.322008,0.139559
4,201011007,Smythes Creek,891,899,1845,601,0.210191,0.212078,0.435244,0.141779


In [87]:
print(out["sa2_name"].sort_values().unique())

['Abbotsford' 'Airport West' 'Albert Park' 'Alexandra' 'Alfredton'
 'Alphington - Fairfield' 'Alps - East' 'Alps - West' 'Altona'
 'Altona Meadows' 'Altona North' 'Ararat' 'Ararat Surrounds'
 'Ardeer - Albion' 'Armadale' 'Ascot Vale' 'Ashburton (Vic.)'
 'Ashwood - Chadstone' 'Aspendale Gardens - Waterways' 'Avoca'
 'Avondale Heights' 'Bacchus Marsh' 'Bacchus Marsh Surrounds' 'Bairnsdale'
 'Ballarat' 'Ballarat East - Warrenheip' 'Ballarat North - Invermay'
 'Balwyn' 'Balwyn North' 'Bannockburn' 'Baranduda - Leneva'
 'Barwon Heads - Armstrong Creek' 'Bayswater' 'Bayswater North'
 'Beaconsfield - Officer' 'Beaufort' 'Beaumaris' 'Beechworth'
 'Belgrave - Selby' 'Belmont' 'Benalla' 'Benalla Surrounds' 'Bendigo'
 'Bendigo Surrounds - North' 'Bendigo Surrounds - South'
 'Bentleigh - McKinnon' 'Bentleigh East - North' 'Bentleigh East - South'
 'Berwick - North' 'Berwick - South East' 'Berwick - South West'
 'Blackburn' 'Blackburn South' 'Boronia' 'Box Hill' 'Box Hill North'
 'Braeside' 'Braybr